# TextCNN — Single-Task Models
Trains two independent TextCNN models, one per target label.  
Each model only sees its own label during training (no shared backbone).  
Compare with `Sprint_2.ipynb` which trains a single multi-task model on both labels simultaneously.  
Train/dev only — test set not touched.

Had help from Claude on implementation.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import pandas as pd
import torch
import cnn_baseline as cnn

from config import DATA_DIR, FASTTEXT_PATH, TARGETS
from preprocess import preprocess
from metrics import compute_metrics, print_confusion_matrix, print_sklearn_report, error_analysis

DEVICE = torch.device(
    "mps"  if torch.backends.mps.is_available()  else
    "cuda" if torch.cuda.is_available()           else
    "cpu"
)
print(f"Device: {DEVICE}")

/Users/jennifer/miniforge3/envs/colx_misinformation/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: mps


In [2]:
# Load data (shared across all models)
train_rows = cnn.load_csv(DATA_DIR / "mis_df_train.csv")
dev_rows   = cnn.load_csv(DATA_DIR / "mis_df_dev.csv")
print(f"Train: {len(train_rows)} | Dev: {len(dev_rows)}")

Train: 600 | Dev: 200


In [3]:
# Build vocab and embeddings once — shared across both models
vocab        = cnn.build_vocab(train_rows, preprocess)
embed_matrix = cnn.load_fasttext_vectors(FASTTEXT_PATH, vocab)
print(f"Vocab size: {len(vocab):,} | Embedding matrix: {tuple(embed_matrix.shape)}")

Loading FastText vectors from /Users/jennifer/.cache/huggingface/hub/datasets--COLX523--fasttext-cc-en-300/snapshots/63ba06b23eb770a6fe0f6c17b0de981e43aec6aa/cc.en.300.vec …
  3875/4091 vocab tokens found in FastText vectors (94.7%)
Vocab size: 4,091 | Embedding matrix: (4091, 300)


In [4]:
# Train one independent model per target
# Each model only optimises for its own label — no shared backbone
results = {}
models  = {}

train_loader = cnn.make_loader(train_rows, vocab, shuffle=True,  tokenize_fn=preprocess)
dev_loader   = cnn.make_loader(dev_rows,   vocab, shuffle=False, tokenize_fn=preprocess)

for target in TARGETS:
    print(f"\n{'='*55}")
    print(f"Training: {target}")
    print(f"{'='*55}")
    model = cnn.TextCNN(len(vocab), embed_matrix).to(DEVICE)
    model = cnn.train_model(model, train_loader, dev_loader, train_rows, DEVICE,
                            train_targets=[target])
    models[target]               = model
    results[f"TextCNN — {target}"] = cnn.predict(model, dev_loader, DEVICE, target=target)


Training: opinion_label
Epoch   1 | loss=0.7467 | avg_dev_f1=0.6976 (opinion_label: 0.6976)
Epoch   2 | loss=0.6497 | avg_dev_f1=0.6928 (opinion_label: 0.6928)
Epoch   3 | loss=0.6010 | avg_dev_f1=0.7093 (opinion_label: 0.7093)
Epoch   4 | loss=0.5371 | avg_dev_f1=0.7014 (opinion_label: 0.7014)
Epoch   5 | loss=0.5012 | avg_dev_f1=0.7177 (opinion_label: 0.7177)
Epoch   6 | loss=0.4437 | avg_dev_f1=0.7172 (opinion_label: 0.7172)
Epoch   7 | loss=0.3909 | avg_dev_f1=0.6900 (opinion_label: 0.6900)
Epoch   8 | loss=0.3446 | avg_dev_f1=0.7118 (opinion_label: 0.7118)
Epoch   9 | loss=0.2789 | avg_dev_f1=0.7261 (opinion_label: 0.7261)
Epoch  10 | loss=0.2521 | avg_dev_f1=0.7159 (opinion_label: 0.7159)
Epoch  11 | loss=0.2244 | avg_dev_f1=0.7081 (opinion_label: 0.7081)
Epoch  12 | loss=0.1804 | avg_dev_f1=0.7081 (opinion_label: 0.7081)
Epoch  13 | loss=0.1683 | avg_dev_f1=0.7166 (opinion_label: 0.7166)
Epoch  14 | loss=0.1379 | avg_dev_f1=0.7144 (opinion_label: 0.7144)
  Early stopping (no im

In [5]:
# Metrics comparison table
rows = []
for name, (preds, labels, probs) in results.items():
    m = compute_metrics(preds, labels, probs)
    rows.append({"Model": name, "Accuracy": m["accuracy"], "Macro F1": m["macro_f1"],
                 "F1 (class 0)": m["f1_class0"], "F1 (class 1)": m["f1_class1"],
                 "AUC-ROC": m.get("auc_roc", float("nan"))})

pd.set_option("display.float_format", "{:.4f}".format)
pd.DataFrame(rows).set_index("Model")

,Accuracy,Macro F1,F1 (class 0),F1 (class 1),AUC-ROC
Model,,,,,
TextCNN — opinion_label,0.7300,0.7261,0.7589,0.6932,0.7930
TextCNN — misinformation_label,0.9050,0.8773,0.9356,0.8190,0.9528


In [6]:
# Per-model detail
for name, (preds, labels, probs) in results.items():
    print(f"\n{'='*50}\n{name}\n{'='*50}")
    print_confusion_matrix(preds, labels)
    print()
    print_sklearn_report(preds, labels)
    print()
    error_analysis(dev_rows, preds, labels)


TextCNN — opinion_label
Confusion matrix (rows=true, cols=predicted):
                 pred=0  pred=1
  true=0 (not-op):    85      32
  true=1 (opinion):   22      61

              precision    recall  f1-score   support

 not-opinion       0.79      0.73      0.76       117
     opinion       0.66      0.73      0.69        83

    accuracy                           0.73       200
   macro avg       0.73      0.73      0.73       200
weighted avg       0.74      0.73      0.73       200


False Positives (predicted opinion, actually not) — 5 shown:
  [17] 'Birds in a blizzard. We put out extra sunflower seeds since their usual food sources just got‚Ä¶ https://www.instagram.c'
  [25] "I'm getting used to seeing a layer of ash on my car each morning - and I'm in AB. We've  only seen a red sun this week. "
  [55] "@jihettly @esd2000 good morning! Yes I'm in the middle of the blizzard. I have plenty of food so I'm happy! Lol. Stay wa"
  [59] 'Re: Hurricane Matthew: All of the @ASUTenni